# Publish full CSA SCST cache from exact Kaggle Output versions

Source ducanh3/notebook7bc83c4c76. Part 0 / 1 / 2 map to Save Versions 82 / 83 / 84. Select CPU, enable Internet and enable Kaggle API Secret codex with access to all source versions and the destination account. Review OWNER and SLUG. The notebook uses exact API downloads; no Add Input is needed.

It verifies all three reviewed receipts before downloading large payloads, validates all ordered IDs and hashes, then creates one private Dataset. It uses /tmp, with at least 40 GiB free required. Four patch arrays are published separately and cache_manifest.json maps the remaining files. Check Dataset processing after API acceptance.


In [ ]:
from pathlib import Path
OWNER='ducanh3'
SLUG='csa-scst-full-karpathy-cache-v1'
TITLE='CSA SCST full Karpathy cache v1'
SECRET_NAME='codex'
SOURCE_NOTEBOOK='ducanh3/notebook7bc83c4c76'
VERSIONS=(82,83,84)
AUDIT=Path('/kaggle/working/csa_scst_full_cache_publish_v82_84')


In [ ]:
import json,os,re,subprocess,sys,tempfile,shutil
from kaggle_secrets import UserSecretsClient
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('Title/slug must have6–50characters')
try:token=UserSecretsClient().get_secret(SECRET_NAME).strip()
except Exception:raise RuntimeError('Enable the named Secret for the owner of Kaggle Dataset upload') from None
if not token or token.startswith('sk-'):raise RuntimeError('Use a Kaggle API token in the named Secret')
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(k,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    process=subprocess.Popen(list(map(str,args)),env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace')
    for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
    if process.wait():raise RuntimeError('API command failed; preserve staged files and read redacted log above')
run_cli([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],authenticated=False)


In [ ]:
EXPECTED=[{'part': 0, 'receipt_name': 'scst_cache_receipt.json', 'receipt_sha256': '3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510', 'archive_sha256': '64eb7d3c958fca4996d717c300191022176e5d688ddd4ba6ea94a5f1a30d3946', 'train_start': 0, 'train_stop': 40000, 'ids_sha256': 'f1fd1507bb3dc6b5dd4ddc57a83e723e2610ec7a45e7a6b05813fa633e519480', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef'}, {'part': 1, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '7ba08f77f38ce48fc8a8a023dbe70ffd6b9bed142f28d54711a9c3ebd0b98a12', 'archive_sha256': 'b9cadfb1f55067b767ed1e32012d5b5bfc7f9f0000c9ff8df0c2ed9676ab461c', 'train_start': 40000, 'train_stop': 80000, 'ids_sha256': '570fab90751d77fda57aebe84a17537a7cba57d3244fc06a5cdcb55ff2c1d885', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef'}, {'part': 2, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '3b8247a0e3fc24aff0a0b2125ce5fbd372976274c5fcf5e1f46315516824ad5c', 'archive_sha256': 'a78982bb44fbb4ce5668d36f9df47de5da9536834ac379f123349a893fd0c9c8', 'train_start': 80000, 'train_stop': 113287, 'ids_sha256': '22d47e2713dbe61777a97a8104ba4bc2490e253c4ff8fd71d0e871d57b2d38fb', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef'}]
EXPECTED_IDS_SHA='6165b0fe756398f3625c4504ebe62f25f07657ce0cffc571a2ce7613ada6e263'

import hashlib, time, zipfile
import numpy as np

def sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

def id_digest(ids):
    return hashlib.sha256(json.dumps(ids,separators=(',',':')).encode()).hexdigest()

def copy_checked(path, target, digest):
    h=hashlib.sha256()
    with path.open('rb') as src:
        while block:=src.read(8*1024**2):
            target.write(block)
            h.update(block)
    if h.hexdigest()!=digest:
        raise ValueError('Changed payload: '+str(path))

if SOURCE_NOTEBOOK!='ducanh3/notebook7bc83c4c76' or VERSIONS!=(82,83,84):
    raise ValueError('This notebook pins exact source versions 82, 83, 84')
if shutil.disk_usage('/tmp').free<40*1024**3:
    raise RuntimeError('Need at least 40 GiB free in /tmp for source download and packaging')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa_scst_download_',dir='/tmp'))
CLI=[sys.executable,'-u','-c','from kaggle.cli import main; main()']
routing = """
import sys, tempfile
from types import SimpleNamespace
from unittest.mock import patch
from kaggle.api.kaggle_api_extended import KaggleApi
api=KaggleApi()
for version in ('82','83','84'):
    with tempfile.TemporaryDirectory() as folder, patch.object(api,'build_kaggle_client') as factory:
        endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output
        endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)
        api.kernels_output('ducanh3/notebook7bc83c4c76/'+version,path=folder)
        req=endpoint.call_args.args[0]
        assert (req.user_name,req.kernel_slug,req.version_label)==('ducanh3','notebook7bc83c4c76','v'+version)
print('Exact version routing mock PASS',flush=True)
"""
run_cli([sys.executable,'-u','-c',routing],authenticated=False)
sources=[]
for spec,version in zip(EXPECTED,VERSIONS,strict=True):
    slug=f'{SOURCE_NOTEBOOK}/{version}'
    target=DOWNLOAD/f'v{version}'
    target.mkdir()
    name=spec['receipt_name']
    run_cli([*CLI,'kernels','output',slug,'-p',target,'--file-pattern',r'(^|/)'+re.escape(name)+r'$'])
    matches=[p for p in target.rglob(name) if sha(p)==spec['receipt_sha256']]
    if len(matches)!=1:
        raise ValueError(f'Output {slug} does not match reviewed cache part {spec["part"]}')
    path=matches[0]
    print('Verified source receipt:',slug,'part',spec['part'],flush=True)
    sources.append((dict(spec,notebook_version=slug),path,target))
# Validate all three identities before downloading any large arrays.
for spec,path,target in sources:
    receipt=json.loads(path.read_text(encoding='utf8'))
    names=list(receipt['files_sha256'])
    pattern=r'(^|/)(?:'+ '|'.join(re.escape(n) for n in names)+r')$'
    run_cli([*CLI,'kernels','output',spec['notebook_version'],'-p',target,'--file-pattern',pattern])

all_ids=[];inputs=[];total=0
for expected,path,_ in sources:
    root=path.parent
    receipt=json.loads(path.read_text(encoding='utf8'))
    size=expected['train_stop']-expected['train_start']
    ids_path=root/'train/pilot_ids.npy'
    ids=np.load(ids_path,allow_pickle=False)
    patches=np.load(root/'train/pilot_patches.npy',mmap_mode='r',allow_pickle=False)
    sequences=np.load(root/'train/pilot_sequences.npy',mmap_mode='r',allow_pickle=False)
    if ids.shape!=(size,) or ids.dtype.kind not in 'iu' or patches.shape!=(size,196,768) or patches.dtype!=np.float16 or sequences.shape!=(size,54):
        raise ValueError('Wrong shard array headers: '+str(root))
    del patches,sequences
    ids=ids.tolist()
    if id_digest(ids)!=expected['ids_sha256']:
        raise ValueError('Changed ordered TRAIN IDs')
    all_ids.extend(ids)
    if expected['part']==0:
        valid_ids=np.load(root/'valid/ids.npy',allow_pickle=False).tolist()
        valid=np.load(root/'valid/patches.npy',mmap_mode='r',allow_pickle=False)
        if valid.shape!=(5000,196,768) or valid.dtype!=np.float16:
            raise ValueError('Wrong VALID patches')
        del valid
    files={}
    for name,digest in receipt['files_sha256'].items():
        p=(root/name).resolve()
        if not p.is_relative_to(root.resolve()) or not p.is_file():
            raise FileNotFoundError(p)
        files[name]=(p,digest)
        total+=p.stat().st_size
    inputs.append((expected,path,files))
if (len(all_ids)!=113287 or len(set(all_ids))!=113287 or id_digest(all_ids)!=EXPECTED_IDS_SHA
        or len(valid_ids)!=5000 or len(set(valid_ids))!=5000 or set(valid_ids)&set(all_ids)):
    raise ValueError('Wrong joined TRAIN/VALID coverage')
bundle_bytes=sum(p.stat().st_size for _,_,files in inputs for name,(p,_) in files.items()
                 if name not in ('train/pilot_patches.npy','valid/patches.npy'))
if shutil.disk_usage('/tmp').free < bundle_bytes+1024**3:
    raise RuntimeError('Insufficient space for small metadata bundle')
if AUDIT.exists():
    raise RuntimeError('Use a fresh AUDIT path to preserve earlier receipts')
AUDIT.mkdir(parents=True)
DESTINATION=Path(tempfile.mkdtemp(prefix='csa_scst_full_upload_',dir='/tmp'))
print('Verified ordered IDs. Staging',round(total/1024**3,2),'GiB in',DESTINATION,flush=True)
manifest=dict(schema='csa_scst_full_dataset_v1',train_images=113287,valid_images=5000,
              test_used=False,ordered_train_ids_sha256=EXPECTED_IDS_SHA,parts=[])
bundle=DESTINATION/'cache_metadata_zip.bin'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for expected,receipt,files in inputs:
        i=expected['part'];receipt_name=f'part{i}/{receipt.name}'
        z.writestr(receipt_name,receipt.read_bytes())
        entry=dict(**expected,receipt_member=receipt_name,files={})
        for name,(path,digest) in files.items():
            member=f'part{i}/{name}'
            if name in ('train/pilot_patches.npy','valid/patches.npy'):
                storage=f'part{i}_'+name.replace('/','_')
                if not path.is_relative_to(DOWNLOAD.resolve()) or sha(path)!=digest:
                    raise ValueError('Changed downloaded patch file: '+str(path))
                os.replace(path,DESTINATION/storage)
                entry['files'][name]=dict(file=storage,sha256=digest)
                print('Verified',storage,flush=True)
            else:
                with z.open(member,'w',force_zip64=True) as out:
                    copy_checked(path,out,digest)
                entry['files'][name]=dict(bundle=bundle.name,member=member,sha256=digest)
        manifest['parts'].append(entry)
manifest['bundle_sha256']=sha(bundle)
for path in (DESTINATION/'cache_manifest.json',AUDIT/'cache_manifest.json'):
    path.write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf8')
print('All input payload SHA checks passed. Ready to upload.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; use fresh destination')
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private CSA SCST cache from three verified CPU Outputs. Karpathy TRAIN113287 and VALID5000. Includes FP16 CLIP patch arrays, region prompt tensors and shared reward/scoring assets. Upstream terms retained.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be active. Check page before retrying.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
(AUDIT/'upload_receipt.json').write_text(json.dumps(dict(status='CREATE_ACCEPTED_PROCESSING_PENDING',
    dataset=f'{OWNER}/{SLUG}',manifest_sha256=sha(AUDIT/'cache_manifest.json')),indent=2),encoding='utf8')

